In [ ]:
import sys
from pathlib import Path
import glob

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Circle
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature

sys.path.insert(0, str(Path.cwd().parent / "src"))
from data import PreprocessedGenotypeDataset, prepare_data_paths, RAW_DATA_DIR
from main import OUTER_FOLDS, INNER_FOLDS, RANDOM_SEED_DATA_SPLIT

# Prepare geography plots

In [ ]:
sample_metadata_df = pd.read_csv(Path(RAW_DATA_DIR) / "igsr-1000 genomes 30x on grch38.tsv", sep="\t")

cls_code_to_name = sample_metadata_df[["Population code", "Population name"]].drop_duplicates().reset_index(drop= True)

In [ ]:
locations = {
    'FIN': (25.0, 60.0),  # Finnish
    'GBR': (-0.1276, 51.5074),  # British,English
    'CHS': (113.7633, 22.18),  # Southern Han Chinese
    'PUR': (-66.1057, 18.4655),  # Puerto Rican
    'CDX': (100.212, 22.0000),  # Dai Chinese
    'CLM': (-74.0817, 4.7110),  # Colombian
    'IBS': (-3.7492, 40.4637),  # Iberian,Spanish
    'KHV': (105.8542, 21.0285),  # Kinh Vietnamese
    'PEL': (-77.0428, -12.0464),  # Peruvian
    'PJL': (74.3587, 31.5204),  # Punjabi
    'ACB': (-61.2225, 10.6918),  # African Caribbean
    'GWD': (-15.3101, 13.4432),  # Gambian Mandinka
    'ESN': (6.6745, 6.7439),  # Esan
    'BEB': (90.4125, 23.8103),  # Bengali
    'MSL': (-11.7383, 7.9647),  # Mende
    'ITU': (78.4867, 17.3850),  # Telugu
    'STU': (80.2707, 13.0827),  # Tamil
    'CEU': (2.3522, 48.8566),  # CEPH
    'YRI': (3.9470, 7.3775),  # Yoruba
    'CHB': (116.4074, 39.9042),  # Han Chinese
    'JPT': (139.6917, 35.6895),  # Japanese
    'LWK': (34.7519, 0.2827),  # Luhya
    'MXL': (-99.1332, 19.4326),  # Mexican Ancestry
    'ASW': (-118.2437, 34.0522),  # African Ancestry SW
    'TSI': (11.2558, 43.7696),  # Toscani
    'GIH': (72.5714, 23.0225)  # Gujarati
}  # Population code -> (longitude, latitude)
color_palette = ["#2A363B", "#355C7D", "#99B898", "#E84A5F", "#FF847C", "#FECEAB"]

In [ ]:
def draw_pie_charts_on_map(cities, labs, save_file_name = None, y_log_trans = False, colorbar=None,
                           radius_legend=None, radius_range=None):
    """
    주어진 도시 목록에 대해 고해상도 세계지도 위에 파이 차트를 그립니다.

    Parameters:
    cities (list of dicts): 도시 정보를 담고 있는 리스트. 각 항목은 다음 키를 가짐:
        - name (str): 도시 이름
        - lon (float): 경도
        - lat (float): 위도
        - value (float): 파이 차트에서 첫 번째 조각의 비율
        - radius (float): 파이 차트의 크기
    """
    start_angle = 0

    # 평면 지도 생성
    fig, ax = plt.subplots(figsize=(15, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_global()

    # High-resolution coastlines and borders
    coastline = cfeature.NaturalEarthFeature('physical', 'coastline', '110m', edgecolor='black', facecolor='none', linewidth=0.5)
    borders = cfeature.NaturalEarthFeature('cultural', 'admin_0_boundary_lines_land', '110m', edgecolor='black', facecolor='none', linewidth=0.2)
    ax.add_feature(coastline)
    ax.add_feature(borders)
    ax.add_feature(cfeature.LAND, color='lightgrey')
    ax.add_feature(cfeature.OCEAN, color='white')

    transform_radius = np.log10 if y_log_trans else (lambda v: v)
    if radius_range is not None:
        low = transform_radius(min(city["radius"] for city in cities))
        high = transform_radius(max(city["radius"] for city in cities))
        span = (high - low) or 1.0
        scale_radius = lambda v: radius_range[0] + (transform_radius(v) - low) / span * (radius_range[1] - radius_range[0])
    else:
        radius_normalize = np.mean([transform_radius(city["radius"]) for city in cities]) / 4
        scale_radius = lambda v: transform_radius(v) / radius_normalize

    for city in cities:
        r = scale_radius(city['radius'])

        if "color" in city:  # circle color is given
            ax.add_patch(Wedge(center=(city['lon'], city['lat']), r=r,
                               theta1=0, theta2=360, edgecolor='black', linewidth=0.6,
                               linestyle=city.get('linestyle', '-'), facecolor=city['color'],
                               transform=ccrs.Geodetic()))
        else:  # theta is given
            if "th2" in city:
                sizes = [city['th1'], city['th2'], 1-city['th1']-city['th2']]  # Proportions for the pie chart
                colors = [color_palette[1], color_palette[2], color_palette[3]]  # Colors for train, validation, and test
            else:
                sizes = [city['th1'], 1 - city['th1']]
                colors = [color_palette[1], color_palette[3]]
            for size, color in zip(sizes, colors):
                angle = 360 * size
                wedge = Wedge(center=(city['lon'], city['lat']), r= r,
                            theta1=start_angle, theta2=start_angle + angle,
                            edgecolor='white', linewidth = 0.5, facecolor=color, transform=ccrs.Geodetic())
                ax.add_patch(wedge)
                start_angle += angle
            # ax.text(city['lon'], city['lat'], city['name'], fontsize=8, ha='center', transform=ccrs.Geodetic())

    # Add legend
    theta_legends = []
    if "th1" in labs:
        theta_legends.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_palette[1], markersize=10, label=labs["th1"]))
        if "th2" in city:
            theta_legends.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_palette[2], markersize=10, label=labs["th2"]))
        theta_legends.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_palette[3], markersize=10, label=labs["th_other"]))
        plt.legend(handles=theta_legends, loc=(0.05,0.05))

    # Lenged for color bar
    if colorbar is not None:
        bar = fig.colorbar(colorbar["mappable"], ax=ax, shrink=0.6, pad=0.01, extend="both",
                           ticks=colorbar.get("ticks"))
        bar.ax.set_yticklabels(colorbar["ticklabels"], fontsize=9)
        bar.set_label(colorbar["label"], fontsize=10)
        bar.outline.set_visible(False)

    # Add legend for circle sizes
    max_radius = max(city['radius'] for city in cities)
    min_radius = min(city['radius'] for city in cities)
    if radius_legend is not None:
        legend_sizes = radius_legend
    elif y_log_trans:
        legend_sizes = np.logspace(np.log10(min_radius), np.log10(max_radius), num=3)
    else:
        legend_sizes = np.arange(100, max_radius + 50, 50)

    gap_between_edges = 3  # Fixed gap between the edges of the circles
    current_y = -57 + (max(len(theta_legends), 3) - 3) * 7

    for i, r in enumerate(legend_sizes):
        r_val = scale_radius(r)
        loc_x = -155.5
        loc_y = current_y + r_val
        current_y += r_val * 2 + gap_between_edges
        circ = Circle((loc_x, loc_y), radius=r_val, edgecolor='black', facecolor='lightgrey', transform=ccrs.Geodetic())
        ax.add_patch(circ)
        plt.text(loc_x + 11, loc_y, f'{round(r):,} {labs["radius"]}', fontsize=10, ha='left', va='center', transform=ccrs.Geodetic())

        
    if save_file_name is not None:
        plt.savefig(save_file_name, dpi = 300)
    plt.show()


# Figure 1a. dataset Description

In [ ]:
target_input = "merged_support3_random_1k_seed_42"

feature_data_path, label_file = prepare_data_paths()

# load data
dataset = PreprocessedGenotypeDataset(feature_data_path / target_input, label_file)
(X, y_original, y), groups, label_mapping, variant_info_df = dataset.get_data()

# outer splits
outer_cv = StratifiedGroupKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_SEED_DATA_SPLIT)
outer_splits = list(outer_cv.split(X, y, groups))

# inner splits
outer_fold = 0
outer_train_idx, outer_test_idx = outer_splits[outer_fold]
print(f"Outer fold {outer_fold}: train n={len(y[outer_train_idx])}, test n={len(y[outer_test_idx])}, train groups={np.unique(groups[outer_train_idx]).size}, test groups={np.unique(groups[outer_test_idx]).size}")

inner_cv = StratifiedGroupKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED_DATA_SPLIT)
inner_splits = list(inner_cv.split(np.zeros(len(outer_train_idx)), y[outer_train_idx], groups[outer_train_idx]))
inner_train_idx, inner_val_idx = inner_splits[0]

In [ ]:
y_original_outer_train = y_original[outer_train_idx]
y_train, y_val = y_original_outer_train[inner_train_idx], y_original_outer_train[inner_val_idx]
y_test = y_original[outer_test_idx]

In [ ]:
total_counts = pd.DataFrame(y_original, columns=["class_label"]).value_counts().reset_index(name='Total_Count')
train_counts = pd.DataFrame(y_train, columns=["class_label"]).value_counts().reset_index(name='Train_Count')
val_counts = pd.DataFrame(y_val, columns=["class_label"]).value_counts().reset_index(name='Val_Count')
test_counts = pd.DataFrame(y_test, columns=["class_label"]).value_counts().reset_index(name='Test_Count')

# Merge the counts into a single dataframe
combined_counts = pd.merge(total_counts, train_counts, on='class_label', how='outer')
combined_counts = pd.merge(combined_counts, val_counts, on='class_label', how='outer')
combined_counts = pd.merge(combined_counts, test_counts, on='class_label', how='outer')

In [ ]:
combined_counts

In [ ]:
cities = []
for idx, row in combined_counts.iterrows():
    code = row['class_label']
    if code in locations:
        lon, lat = locations[code]
        city = {
            'name': code,
            'lon': lon,
            'lat': lat,
            'th1': row["Train_Count"] / row["Total_Count"],  # This is an arbitrary placeholder, adjust as necessary
            'th2': row["Val_Count"] / row["Total_Count"],  # This is an arbitrary placeholder, adjust as necessary
            'radius': row["Total_Count"],#  # Normalize by the largest possible value for visualization scaling
        }
        cities.append(city)
    else:
        print(code)


draw_pie_charts_on_map(cities, labs = {"th1": "Train", "th2" : "Validation", "th_other" : "Test", "radius" : "samples"}, save_file_name="results/data_description.pdf")

# Figure 3a. Number of features required for classifying each class (minimum_SNPs)

In [ ]:
df_minimum_n_by_cls = pd.read_csv("results/minimum_n_by_cls.csv")

In [ ]:
N_FOLDS = 5
N_PERMUTATIONS = 4000
PERMUTATION_CHUNK = 500
SEED = 0

populations = sorted(df_minimum_n_by_cls["class_target"])
load_importance = lambda fold, pop: np.load(f"../cache/8192_seed42_cv42_outerfold{fold}_xgb_cls_{pop}_basic_feature_importance_mean.npy")


def weighted_jaccard(a, b):
    """Ruzicka similarity = sum(min) / sum(max)."""
    return np.minimum(a, b).sum(axis=-1) / np.maximum(a, b).sum(axis=-1)


records = []
rng = np.random.default_rng(SEED)
for fold in range(N_FOLDS):
    importance = {pop: load_importance(fold, pop) for pop in populations}
    for i, pop_a in enumerate(populations):
        vector_a = importance[pop_a]
        permuted = np.concatenate([rng.permuted(np.tile(vector_a, (PERMUTATION_CHUNK, 1)), axis=1)
                                   for _ in range(N_PERMUTATIONS // PERMUTATION_CHUNK)])
        for pop_b in populations[i + 1:]:
            observed = weighted_jaccard(vector_a, importance[pop_b])
            null = weighted_jaccard(permuted, importance[pop_b])
            records.append({"outer_fold": fold, "pop_a": pop_a, "pop_b": pop_b,
                            "log2_enrichment": np.log2(observed / null.mean())})
        del permuted
    print(f"  fold {fold} done")

pairwise_similarity = (pd.DataFrame(records).groupby(["pop_a", "pop_b"])
                       .log2_enrichment.agg(["mean", "std"])
                       .rename(columns={"mean": "log2_enrichment", "std": "log2_sd"})
                       .reset_index())
pairwise_similarity["fold_enrichment"] = 2 ** pairwise_similarity.log2_enrichment
pairwise_similarity.to_csv("results/pairwise_SNP_importance_similarity.csv", index=False)


In [ ]:
REFERENCE = "PEL"
POOL_SIZE = 8192
COLOR_LIMIT = 1.5

reference_rows = pairwise_similarity[(pairwise_similarity.pop_a == REFERENCE) |
                                     (pairwise_similarity.pop_b == REFERENCE)].copy()
reference_rows["class_target"] = np.where(reference_rows.pop_a == REFERENCE,
                                          reference_rows.pop_b, reference_rows.pop_a)
similarity_to_reference = reference_rows.set_index("class_target")[["log2_enrichment", "log2_sd", "fold_enrichment"]]
print(similarity_to_reference.sort_values("fold_enrichment", ascending=False).round(3))

diverging = LinearSegmentedColormap.from_list("similarity", [color_palette[1], "#f0efec", color_palette[3]])
norm = TwoSlopeNorm(vcenter=0, vmin=-COLOR_LIMIT, vmax=COLOR_LIMIT)

cities = []
for idx, row in df_minimum_n_by_cls.iterrows():
    code = row["class_target"]
    not_reached = pd.isna(row["select_n"])
    cities.append({
        "name": code, "lon": locations[code][0], "lat": locations[code][1],
        "radius": POOL_SIZE if not_reached else row["select_n"],
        "color": "white" if code == REFERENCE else diverging(norm(similarity_to_reference.log2_enrichment[code])),
        # "linestyle": (0, (2, 1.4)) if not_reached else "-",
    })

draw_pie_charts_on_map(
    cities, y_log_trans=True, labs={"radius": "SNPs"},
    radius_legend=[1024, 2048, 4096, 8192], radius_range=(2.0, 5.0),
    colorbar={"mappable": plt.cm.ScalarMappable(norm=norm, cmap=diverging),
              "ticks": [-1.5, -1, -0.5, 0, 0.5, 1, 1.5],
              "ticklabels": ["0.35x", "0.5x", "0.7x", "1x\n(random)", "1.4x", "2x", "2.8x"],
              "label": f"SNP importance similarity to {REFERENCE}\n(relative to random expectation)",
             },
    save_file_name="results/SNP_overlap.pdf")


In [ ]:
# Supplementary
SUPERPOP_ORDER = ["AFR", "AMR", "EAS", "EUR", "SAS"]
CM = 1 / 2.54
FONT_SIZE = 10

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "pdf.fonttype": 42,   # TrueType embed (editable in Illustrator)
    "ps.fonttype": 42,
})

superpopulation = (sample_metadata_df.drop_duplicates("Population code")
                   .set_index("Population code")["Superpopulation code"])
ordered = sorted(populations, key=lambda p: (SUPERPOP_ORDER.index(superpopulation[p]), p))
position = {pop: i for i, pop in enumerate(ordered)}

matrix = np.full((len(ordered), len(ordered)), np.nan)
for row in pairwise_similarity.itertuples():
    i, j = position[row.pop_a], position[row.pop_b]
    matrix[i, j] = matrix[j, i] = row.log2_enrichment

diverging.set_bad("#d9d8d4")
group_sizes = [sum(superpopulation[p] == s for p in ordered) for s in SUPERPOP_ORDER]
boundaries = np.cumsum(group_sizes)[:-1] - 0.5

fig, ax = plt.subplots(figsize=(21 * CM, 17.5 * CM))
fig.subplots_adjust(left=0.05, right=0.95, bottom=0.07, top=0.95)
image = ax.imshow(np.ma.masked_invalid(matrix), cmap=diverging, norm=norm)
for b in boundaries:
    ax.axhline(b, color="white", lw=1.2)
    ax.axvline(b, color="white", lw=1.2)
ax.set_xticks(range(len(ordered)), ordered, rotation=90, fontsize=FONT_SIZE)
ax.set_yticks(range(len(ordered)), ordered, fontsize=FONT_SIZE)
ax.tick_params(length=0, pad=1.5)
for name, start, size in zip(SUPERPOP_ORDER, np.r_[0, np.cumsum(group_sizes)[:-1]], group_sizes):
    ax.text(start + size / 2 - 0.5, -1.0, name, ha="center", va="bottom",
            fontsize=FONT_SIZE + 1, color=color_palette[0])
for spine in ax.spines.values():
    spine.set_visible(False)
bar = fig.colorbar(image, ax=ax, shrink=0.72, pad=0.02, extend="both",
                   ticks=[-1.5, -1, -0.5, 0, 0.5, 1, 1.5])
bar.ax.set_yticklabels(["0.35x", "0.5x", "0.7x", "1x\n(random)", "1.4x", "2x", "2.8x"], fontsize=FONT_SIZE - 1)
bar.ax.tick_params(length=2, pad=2)
bar.set_label("SNP importance similarity\n(relative to random expectation)", fontsize=FONT_SIZE)
bar.outline.set_visible(False)
plt.savefig("results/Supp_SNP_similarity_matrix.pdf", dpi=300)
plt.show()

within = pairwise_similarity[pairwise_similarity.pop_a.map(superpopulation) ==
                             pairwise_similarity.pop_b.map(superpopulation)]
between = pairwise_similarity[pairwise_similarity.pop_a.map(superpopulation) !=
                              pairwise_similarity.pop_b.map(superpopulation)]
print(f"within {2**within.log2_enrichment.mean():.2f}x | between {2**between.log2_enrichment.mean():.2f}x "
      f"| difference {within.log2_enrichment.mean() - between.log2_enrichment.mean():+.3f} log2")


# Figure 3C. Confusion matrix

In [ ]:
class_order = ['CEU', 'GWD', 'LWK', 'TSI', 'YRI', 'JPT', 'MSL', 'BEB', 'FIN', 'KHV', 'PEL', 'CLM', 'PUR',
               'CDX', 'IBS', 'CHS', 'PJL', 'STU', 'ITU', 'GIH', 'CHB', 'ACB', 'MXL', 'ESN', 'ASW', 'GBR']
confusion_matrix_files = sorted(glob.glob(
    "../results/plots/43034818_seed42_cv42_outerfold*_random_1000000_1000000_random_1000000_SVM_fixed_test_confusion_matrix.csv"
))

conf_matrix = sum(pd.read_csv(f, index_col=0).loc[class_order, class_order] for f in confusion_matrix_files)
print(f"{len(confusion_matrix_files)} folds, {conf_matrix.to_numpy().sum()} samples, "
      f"accuracy = {np.trace(conf_matrix.to_numpy()) / conf_matrix.to_numpy().sum():.3f}")

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

plt.figure(figsize=(12, 10))
cmap = sns.light_palette('#355C7D', as_cmap=True)
sns.heatmap(
    conf_matrix, annot=True, fmt='.0f', cmap=cmap, annot_kws={"size": 12},
    # xticklabels=class_names, yticklabels=class_names,
)
plt.xlabel('Predicted', fontsize=20)
plt.ylabel('Actual', fontsize=20)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.title('Confusion Matrix', fontsize=20)
plt.savefig(f"results/fig3c_confusion_matrix.pdf", format='pdf', dpi = 300)
plt.close()